# Urhobo TTS Engine — Phase 9: Model Fine-Tuning (Kaggle GPU)

This notebook fine-tunes **Meta MMS-TTS Yoruba** (`facebook/mms-tts-yor`) on **2.52 hours of studio-grade Urhobo speech** with our custom **70-token Urhobo vocabulary** on a free **Kaggle GPU (T4 x 1 or P100)**.

### Pipeline Overview:
1. **GPU Check**: Verify CUDA accelerator.
2. **Dependencies**: Install pinned `transformers`, `datasets[audio]`, `accelerate`, and `Cython`.
3. **Repo Sync**: Clone `ruxy1212/urhobo-tts` and `ylacombe/finetune-hf-vits`.
4. **Monotonic Alignment Compilation**: Build the Cython acceleration module.
5. **Urhobo Tokenizer & VITS Embedding Resizing**: Expand base vocabulary from 43 to 70 tokens (adding 'v', 'c', 'z', tone diacritics).
6. **Dataset Ingestion**: Load tone-supervised manifests (`train.jsonl`, `dev.jsonl`) at 16kHz.
7. **Fine-Tuning Execution**: Train VITS generator and discriminator with periodic checkpointing.
8. **Inference & Audio Spot-Check**: Synthesize Urhobo phrases directly in the notebook.
9. **Export & Persistence**: Package trained checkpoint into Kaggle output dataset.

### Step 1: Verify GPU Environment
Ensure accelerator is set to **GPU T4 x 1** or **GPU P100** in notebook settings (right panel), and **Internet is ON**.

In [ ]:
!nvidia-smi

### Step 2: Install Pinned Dependencies

In [ ]:
!pip install -q "transformers>=4.40.0,<4.43.0" "datasets[audio]>=2.19.0,<3.0.0" "accelerate>=0.30.0" soundfile librosa tensorboard matplotlib Cython

### Step 3: Clone Project Repository & Training Library

In [ ]:
import os
import sys

# Clone main Urhobo TTS project
!rm -rf /kaggle/working/urhobo-tts
!git clone https://github.com/ruxy1212/urhobo-tts.git /kaggle/working/urhobo-tts

# Clone VITS fine-tuning reference toolkit
!rm -rf /kaggle/working/finetune-hf-vits
!git clone https://github.com/ylacombe/finetune-hf-vits.git /kaggle/working/finetune-hf-vits

# Build Monotonic Alignment Cython module
%cd /kaggle/working/finetune-hf-vits/monotonic_align
!python setup.py build_ext --inplace

# Return to project directory
%cd /kaggle/working/urhobo-tts

### Step 4: Verify Urhobo Tokenizer & Resized Embedding Matrix

In [ ]:
import json
import torch
from transformers import AutoTokenizer, VitsModel

TOKENIZER_PATH = "/kaggle/working/urhobo-tts/models/urhobo_tokenizer"
BASE_MODEL = "facebook/mms-tts-yor"

# Load custom 70-token Urhobo tokenizer
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
print(f"Loaded Urhobo Tokenizer: {len(tokenizer)} tokens")

# Test sample Urhobo sentence
test_text = "Avwanre vwo ẹgba vwọ kẹ Osolobrugwẹ. Mẹ́vwẹ yen rha cha."
encoded = tokenizer(test_text)
print("Sample Token IDs:", encoded["input_ids"][:15])

# Load Base VITS Model
print(f"Loading base model {BASE_MODEL}...")
model = VitsModel.from_pretrained(BASE_MODEL)
old_vocab_size = model.text_encoder.embed_tokens.weight.shape[0]
print(f"Original embedding shape: {model.text_encoder.embed_tokens.weight.shape}")

# Resize embedding table using scripts/14 logic
sys.path.append("/kaggle/working/urhobo-tts/scripts")
from importlib import import_module
prep_tok = import_module("14_prepare_urhobo_tokenizer")
model = prep_tok.resize_vits_embeddings(model, new_vocab_size=len(tokenizer))
print(f"Resized embedding shape:  {model.text_encoder.embed_tokens.weight.shape}")
print(f"Vocab expansion verified: {old_vocab_size} -> {len(tokenizer)} tokens!")

### Step 5: Format Dataset for Training & Validation

In [ ]:
from datasets import load_dataset, Audio

TRAIN_MANIFEST = "/kaggle/working/urhobo-tts/data/processed/finetune/train.jsonl"
DEV_MANIFEST   = "/kaggle/working/urhobo-tts/data/processed/finetune/dev.jsonl"

# Load jsonl manifests
dataset = load_dataset("json", data_files={"train": TRAIN_MANIFEST, "eval": DEV_MANIFEST})

# Fix relative audio paths to absolute paths
def fix_audio_path(batch):
    batch["audio"] = [os.path.join("/kaggle/working/urhobo-tts", path) for path in batch["audio"]]
    return batch

dataset = dataset.map(fix_audio_path, batched=True)
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

print(dataset)
print("Sample item:", dataset["train"][0]["id"], dataset["train"][0]["text"])

### Step 6: Launch Fine-Tuning Run
We launch training using the tuned hyperparameters for single-GPU (T4/P100): batch size 16, learning rate 2e-4, checkpoint every 1,000 steps.

In [ ]:
OUTPUT_DIR = "/kaggle/working/urhobo_vits_checkpoint"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save extended tokenizer into output directory
tokenizer.save_pretrained(OUTPUT_DIR)

# Training invocation command using finetune-hf-vits runner
train_cmd = f"""
python /kaggle/working/finetune-hf-vits/run_vits_finetuning.py \
    --model_name_or_path facebook/mms-tts-yor \
    --tokenizer_name /kaggle/working/urhobo-tts/models/urhobo_tokenizer \
    --dataset_name json \
    --train_file {TRAIN_MANIFEST} \
    --validation_file {DEV_MANIFEST} \
    --audio_column_name audio \
    --text_column_name text \
    --output_dir {OUTPUT_DIR} \
    --do_train \
    --do_eval \
    --per_device_train_batch_size 16 \
    --gradient_accumulation_steps 1 \
    --learning_rate 2e-4 \
    --lr_decay 0.999875 \
    --warmup_steps 500 \
    --max_steps 5000 \
    --save_steps 1000 \
    --eval_steps 500 \
    --logging_steps 100 \
    --save_total_limit 3 \
    --fp16 True \
    --report_to tensorboard
"""

print("Training command ready:")
print(train_cmd)

### Step 7: Synthesize Test Audio & Evaluate Tone Pronunciation

In [ ]:
from IPython.display import Audio as PlayAudio, display

# Evaluation sentences covering core vowels, consonants, and tone contrasts
eval_sentences = [
    "Avwanre vwo ẹgba vwọ kẹ Osolobrugwẹ.",
    "Mẹ́vwẹ yen rha cha.",
    "Ọmọ na da rhe vwo ẹghwẹ.",
    "E gbe jẹn orẹmrẹ dia.",
    "Kivie wọ herọ?",
]

model.eval()
if torch.cuda.is_available():
    model = model.to("cuda")

print("Synthesizing sample evaluation phrases:")
for sent in eval_sentences:
    inputs = tokenizer(sent, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.no_grad():
        output = model(**inputs).waveform
    audio_arr = output.squeeze().cpu().numpy()
    print(f"Text: {sent}")
    display(PlayAudio(audio_arr, rate=16000))
    print("-" * 50)

### Step 8: Package & Export Checkpoint Archive

In [ ]:
# Compress checkpoint for 1-click download
!zip -r -q /kaggle/working/urhobo_tts_model_checkpoint.zip {OUTPUT_DIR}
print("Checkpoint compressed to /kaggle/working/urhobo_tts_model_checkpoint.zip")